In [ ]:
import os
import sys
import yaml
import numpy as np
import random
import time
import datetime
import copy
import matplotlib.pyplot as plt
from scipy.optimize import minimize, basinhopping
from scipy.signal import find_peaks
from neuron import h, gui
from specify_cells import CA1_Pyr, QuickSim
import csv


import yaml
from optimization_utils import * 
from plot_utils import *


In [ ]:
def assign_exc_and_inh_synapse_stims(num_exc_syns, excitatory_stochastic, num_inh_syns):
    # Dictionaries to store the actual synapse objects
    stim_exc_syns = {'CA3': [], 'ECIII': []}
    stim_inh_syns = {'CA3': [], 'ECIII': []}

    # 1. Place Excitatory Synapses
    for pathway, num_to_insert in num_exc_syns.items():
        list_of_valid_syn_locs = []
        if pathway == 'ECIII':
            list_of_valid_syn_locs.extend(exc_syn_locs_by_sec_type['tuft'])
        else:
            for sec_type in ['trunk', 'apical']:
                list_of_valid_syn_locs.extend(exc_syn_locs_by_sec_type[sec_type])

        actual_num = min(int(num_to_insert), len(list_of_valid_syn_locs))
        selected_locs = local_random.sample(list_of_valid_syn_locs, actual_num)
        
        syn_list = cell.insert_synapses_at_syn_locs(selected_locs, exc_syn_types, stochastic=excitatory_stochastic)
        stim_exc_syns[pathway].extend(syn_list)
    
    # 2. Place Inhibitory Synapses
    for pathway, num_to_insert in num_inh_syns.items():
        list_of_valid_syn_locs = []
        if pathway == 'ECIII':
            list_of_valid_syn_locs.extend(inh_syn_locs_by_sec_type['tuft'])
        else:
            for sec_type in ['trunk', 'apical']:
                list_of_valid_syn_locs.extend(inh_syn_locs_by_sec_type[sec_type])

        actual_num = min(int(num_to_insert), len(list_of_valid_syn_locs))
        selected_locs = local_random.sample(list_of_valid_syn_locs, actual_num)
        
        syn_list = cell.insert_synapses_at_syn_locs(selected_locs, inh_syn_types, stochastic=False)
        stim_inh_syns[pathway].extend(syn_list)

    return stim_exc_syns, stim_inh_syns 


In [ ]:
target_plateau_per_cycle = [3.337, 3.5297, 4.356, 5.2235, 5.6729]
target_spikes_per_cycle  = [6, 6, 7, 7, 7]
target_trough_per_cycle  = [1.2946, 3.1996, 8.0606, 17.4871, 22.5451]

# Optimization Configuration
#config_path = 'optimization_config_BasinHopping_20260322_4th_Run_Overnight.yaml'
config_path = 'optimization_config_BasinHopping_20260324_5th_Run_Overnight.yaml' 
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

In [ ]:
# Simulation parameters
duration  = config['simulation']['equilibrate'] + config['simulation']['sim_duration']
dt        = config['simulation']['dt']
v_init    = config['simulation']['v_init']
tbs_times = config['simulation']['tbs_times']



### Build a CA1 cell with the same synapse locations

In [ ]:
print("--- Initializing Cell and Synaptic Infrastructure ---")
NMDA_type = 'NMDA_KIN5'
exc_syn_types = ['AMPA_KIN', NMDA_type]
#inh_syn_types = ['GABA_A_KIN'] #simulate blocking GABA Receptors with Gabazine
inh_syn_types = ['GABAb']

# SYNAPSE_SEED must match optimize_soma_plateaus.py SYNAPSE_SEED=0
SYNAPSE_SEED = 0
local_random = random.Random(SYNAPSE_SEED)

cell = CA1_Pyr(morph_filename=config['simulation']['morph_filename'],
               mech_filename=config['simulation']['mech_filename'],
               full_spines=False)

exc_syns_sec_types = ['trunk', 'apical', 'tuft']
inh_syns_sec_types = ['trunk', 'apical', 'tuft']
stim_successes = []

exc_syn_locs_by_sec_type = cell.get_excitatory_syn_locs(sec_type_list=exc_syns_sec_types)
inh_syn_locs_by_sec_type = cell.get_inhibitory_syn_locs(sec_type_list=inh_syns_sec_types)


In [ ]:
# --- 1. SET COUNTS (match YAML config) ---
num_exc_syns = {'CA3': 100, 'ECIII': 160}
num_inh_syns = {'CA3': 120, 'ECIII': 120}
excitatory_stochastic = True

# --- 2. CRITICAL RESET ---
for node in cell.tree:
    node.content['synapses'] = []
local_random.seed(SYNAPSE_SEED)

# --- 3. ASSIGN ---
stim_exc_syns, stim_inh_syns = assign_exc_and_inh_synapse_stims(num_exc_syns, excitatory_stochastic, num_inh_syns)

# --- 4. INITIALIZE MECHANISMS ---
cell.init_synaptic_mechanisms()

# --- 5. APPLY I80T MUTATION (if using the I80T YAML) ---
import yaml
with open('data/' + config['simulation']['mech_filename'], 'r') as f:
    biophys_config = yaml.safe_load(f)

gabab_mut = biophys_config.get('gabab_mutation', None)
if gabab_mut:
    gmax_mult = gabab_mut['gmax_mult']
    for pathway in stim_inh_syns:
        for syn in stim_inh_syns[pathway]:
            if 'GABAb' in syn._syn:
                syn._syn['GABAb']['target'].gmax *= gmax_mult
    print(f"I80T mutation applied: GABAb gmax × {gmax_mult}")
else:
    print("No gabab_mutation field found — running as WT")

print(f"Assigned {sum(len(v) for v in stim_exc_syns.values())} Exc and {sum(len(v) for v in stim_inh_syns.values())} Inh synapses.")


In [ ]:
%matplotlib inline 

In [ ]:
plot_mech_param_distribution(cell, 'cal', 'gcalbar',  param_label = 'Low-Threshold L-type Calcium Channels') 

In [ ]:
plot_mech_param_distribution(cell, 'calH', 'gcalbar', param_label = 'High-Threshold L-type Calcium Channels') 

In [ ]:
plot_mech_param_distribution(cell, 'cat', 'gcatbar', param_label = 'T-type Calcium Channels') 

In [ ]:
plot_mech_param_distribution(cell, 'car', 'gcabar', param_label = 'R-type Calcium Channels') 

# R-type Calcium channels are known to contribute significantly to dendritic spike regeneration in CA1 pyramidal cells (often contributing about 20-30% of the total Calcium influx during a spike). 
# By distributing them uniformly, the neuron has a "baseline" excitability for back-propagating action potentials (bAPs) 
# or local dendritic spikes at any point in the tree, without favoring the distal tuft as much as the T-type channels do.

In [ ]:
plot_mech_param_distribution(cell, 'mykca', 'gkbar', param_label = 'BK channels') # Calcium activated mAHP K channel - BK.

In [ ]:
plot_mech_param_distribution(cell, 'kca', 'gbar', param_label = 'SK channels') # Calcium activated sAHP K channel - SK.

In [ ]:
# To plot GABAb gmax (Peak Conductance) across the dendritic tree
plot_synaptic_param_distribution(cell, 'GABAb', 'gmax', scale_factor=1.0)


In [ ]:
trunk_bifurcation = [trunk for trunk in cell.trunk if cell.is_bifurcation(trunk, 'trunk')]
if trunk_bifurcation:
    trunk_branches = [branch for branch in trunk_bifurcation[0].children if branch.type == 'trunk']
    # get where the thickest trunk branch gives rise to the tuft
    trunk = max(trunk_branches, key=lambda node: node.sec(0.).diam)
    trunk = next((node for node in cell.trunk if cell.node_in_subtree(trunk, node) and 'tuft' in (child.type
                                                                                    for child in node.children)))
else:
    trunk_bifurcation = [node for node in cell.trunk if 'tuft' in (child.type for child in node.children)]
    trunk = trunk_bifurcation[0]

In [ ]:
sim = QuickSim(duration, cvode=False, dt=dt, verbose=0)
sim.parameters['equilibrate'] = config['simulation']['equilibrate'] 

sim.parameters['duration'] = config['simulation']['sim_duration'] 
sim.parameters['stim_dt'] = config['simulation']['dt'] 
sim.append_rec(cell, cell.tree.root, description='soma', loc=0.)
sim.append_rec(cell, trunk_bifurcation[0], description='proximal_trunk', loc=1.)
sim.append_rec(cell, trunk, description='distal_trunk', loc=1.)
spike_output_vec = h.Vector()
cell.spike_detector.record(spike_output_vec)

stim_t = np.arange(-config['simulation']['equilibrate'] , config['simulation']['sim_duration'] , config['simulation']['dt'] )
tbs_vec = h.Vector(tbs_times)

# 2. Stimulate both CA3 and ECIII
for group in stim_exc_syns:
    for syn in stim_exc_syns[group]:
        syn.source.play(tbs_vec)

for group in stim_inh_syns:
    for syn in stim_inh_syns[group]:
        syn.source.play(tbs_vec)



In [ ]:
print(f"Starting simulation for {duration} ms...")
sim.run(v_init=v_init)
print("Simulation complete.")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), dpi=100)
fig.patch.set_facecolor('white')

# --- PLOT SOMA (Black) ---
for rec in sim.rec_list:
    if 'soma' in rec['description'].lower():
        ax1.plot(sim.tvec.to_python(), rec['vec'].to_python(), color='black', lw=1.2)
        ax1.set_xlim(400, 1500)

# --- PLOT DISTAL DENDRITE ONLY (Black) ---
for rec in sim.rec_list:
    desc = rec['description'].lower()
    # Specifically looking for 'distal' and 'trunk' (or just 'distal')
    if 'distal' in desc and 'trunk' in desc:
        ax2.plot(sim.tvec.to_python(), rec['vec'].to_python(), color='black', lw=1.2)
        ax2.set_xlim(400, 1500)

# --- FUNCTION TO ADD SCALE BAR ---
def add_scalebar(ax, scale_x=100, scale_y=20):
    xlim = ax.get_xlim()
    ylim = ax.get_ylim()
    
    # Clean up axes
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.set_xticks([])
    ax.set_yticks([])
    
    # Position scale bar
    x_pos = xlim[1] - scale_x - 50 
    y_pos = ylim[0] + (ylim[1] - ylim[0]) * 0.05 
    
    # Draw bars
    ax.plot([x_pos, x_pos + scale_x], [y_pos, y_pos], color='black', lw=2, solid_capstyle='butt')
    ax.plot([x_pos, x_pos], [y_pos, y_pos + scale_y], color='black', lw=2, solid_capstyle='butt')
    
    # Add labels
    ax.text(x_pos + scale_x/2, y_pos - (ylim[1] - ylim[0]) * 0.02, f'{scale_x} ms', ha='center', va='top')
    ax.text(x_pos - (xlim[1] - xlim[0]) * 0.01, y_pos + scale_y/2, f'{scale_y} mV', ha='right', va='center', rotation=90)

# Apply scale bars
add_scalebar(ax1)
add_scalebar(ax2)

plt.tight_layout()
# plt.savefig('distal_dendrite_traces_GIRK_LOF.svg', format='svg', bbox_inches='tight')
plt.show()